# Digital Image Processing Lab - Assignment 2
**Problem Statement:**
Write a program to convert RGB (Color) Image to grayscale image:
- **Case 1:** By taking mean average of three planes.
- **Case 2:** Input the weightage of three planes, i.e., $R, G, B$ from the user. The weightage is a value between $0$ and $1$, and sum of all three weightages is equal to $1$. *(Example: Weightage of R, G, B is 0.7, 0.2, 0.1, respectively).*

---
### Mathematical Formulation
An RGB color image consists of three color planes: Red $R(x, y)$, Green $G(x, y)$, and Blue $B(x, y)$.

1. **Case 1: Mean Average Method:**
   $$I_{\text{gray}}(x, y) = \frac{R(x, y) + G(x, y) + B(x, y)}{3}$$

2. **Case 2: User-Defined Weighted Method:**
   $$I_{\text{gray}}(x, y) = w_R \cdot R(x, y) + w_G \cdot G(x, y) + w_B \cdot B(x, y)$$
   Subject to constraints:
   $$0 \le w_R, w_G, w_B \le 1 \quad \text{and} \quad w_R + w_G + w_B = 1$$

3. **Standard Benchmark (ITU-R BT.601 / NTSC Luminosity):**
   Human eyes are most sensitive to green, moderately to red, and least to blue. The standard standard luminance formula used by OpenCV is:
   $$I_{\text{NTSC}} = 0.299 \cdot R + 0.587 \cdot G + 0.114 \cdot B$$


In [ ]:
# Cell 1: Import Libraries
import numpy as np
import matplotlib.pyplot as plt
import cv2

plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# Cell 2: Image Loader & Synthetic Color Image Generator
def get_sample_rgb_image():
    """Generates a vibrant RGB test image with distinct primary and secondary colors."""
    h, w = 300, 300
    img = np.zeros((h, w, 3), dtype=np.uint8)
    
    # Red circle
    cv2.circle(img, (100, 100), 70, (255, 0, 0), -1)
    # Green circle
    cv2.circle(img, (200, 100), 70, (0, 255, 0), -1)
    # Blue circle
    cv2.circle(img, (150, 190), 70, (0, 0, 255), -1)
    # Yellow rectangle (R+G)
    cv2.rectangle(img, (30, 230), (120, 280), (255, 255, 0), -1)
    # Cyan rectangle (G+B)
    cv2.rectangle(img, (180, 230), (270, 280), (0, 255, 255), -1)
    
    # Blend overlaps
    return img

def load_rgb_image(filepath=None):
    if filepath and cv2.haveImageReader(filepath):
        bgr = cv2.imread(filepath)
        if bgr is not None:
            return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    print("Using built-in vibrant RGB test pattern.")
    return get_sample_rgb_image()

rgb_img = load_rgb_image()
print(f"RGB Image Loaded: Shape={rgb_img.shape}, Type={rgb_img.dtype}")


In [ ]:
# Cell 3: Case 1 - Mean Average Method
def rgb_to_gray_average(rgb):
    """Converts RGB to grayscale using mean average of R, G, B."""
    # Cast to float to avoid overflow during sum
    r = rgb[:, :, 0].astype(np.float32)
    g = rgb[:, :, 1].astype(np.float32)
    b = rgb[:, :, 2].astype(np.float32)
    gray = (r + g + b) / 3.0
    return np.clip(np.round(gray), 0, 255).astype(np.uint8)

gray_case1 = rgb_to_gray_average(rgb_img)
print(f"Case 1 Grayscale generated: range [{gray_case1.min()}, {gray_case1.max()}]")


In [ ]:
# Cell 4: Case 2 - User-Defined Weighted Method
# User configurable weights
w_r = 0.7 #@param {type:"number"}
w_g = 0.2 #@param {type:"number"}
w_b = 0.1 #@param {type:"number"}

def rgb_to_gray_weighted(rgb, wr, wg, wb):
    """Converts RGB to grayscale with weights wr, wg, wb."""
    total = wr + wg + wb
    if not np.isclose(total, 1.0, atol=1e-3):
        print(f"Warning: Sum of weights ({total}) != 1. Normalizing weights...")
        wr, wg, wb = wr / total, wg / total, wb / total
    
    r = rgb[:, :, 0].astype(np.float32)
    g = rgb[:, :, 1].astype(np.float32)
    b = rgb[:, :, 2].astype(np.float32)
    gray = wr * r + wg * g + wb * b
    return np.clip(np.round(gray), 0, 255).astype(np.uint8), (wr, wg, wb)

gray_case2, normalized_weights = rgb_to_gray_weighted(rgb_img, w_r, w_g, w_b)
print(f"Using normalized weights: R={normalized_weights[0]:.3f}, G={normalized_weights[1]:.3f}, B={normalized_weights[2]:.3f}")

# Reference: ITU-R BT.601 standard luminance
gray_ntsc, _ = rgb_to_gray_weighted(rgb_img, 0.299, 0.587, 0.114)


In [ ]:
# Cell 5: Visualization & Comparison
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Row 1: RGB components
axes[0, 0].imshow(rgb_img)
axes[0, 0].set_title("Original RGB Image", fontsize=13, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(rgb_img[:, :, 0], cmap='Reds')
axes[0, 1].set_title("Red Channel (R)", fontsize=12, fontweight='bold')
axes[0, 1].axis('off')

axes[0, 2].imshow(rgb_img[:, :, 1], cmap='Greens')
axes[0, 2].set_title("Green Channel (G)", fontsize=12, fontweight='bold')
axes[0, 2].axis('off')

# Row 2: Grayscale conversions
axes[1, 0].imshow(rgb_img[:, :, 2], cmap='Blues')
axes[1, 0].set_title("Blue Channel (B)", fontsize=12, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(gray_case1, cmap='gray')
axes[1, 1].set_title("Case 1: Mean Average ((R+G+B)/3)", fontsize=12, fontweight='bold')
axes[1, 1].axis('off')

axes[1, 2].imshow(gray_case2, cmap='gray')
axes[1, 2].set_title(f"Case 2: Custom Weighted\n({normalized_weights[0]:.2f}R + {normalized_weights[1]:.2f}G + {normalized_weights[2]:.2f}B)", fontsize=12, fontweight='bold')
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()


### Observations & Analysis
- **Mean Average:** Weights all channels equally ($33.3\%$ each). While simple, it doesn't account for human photoreceptor spectral sensitivity.
- **Weighted Method:** By assigning $0.7$ to Red, $0.2$ to Green, and $0.1$ to Blue, red areas appear significantly brighter than blue areas in the resulting grayscale output.
- Custom weightage is critical in applications like infrared/thermal vision or color-filtering where specific spectral bands must be emphasized.
